<a target="_blank" href="https://colab.research.google.com/github/agensflow-ai/agensflow-langgraph/blob/main/notebooks/token_budgets_workshop.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# From Token Maxing to Token Budgeting — workshop demo

**TwoSetAI Workshop #3 · 2026-08-14**

This notebook replays 780 real Claude Code + `agent-skills-main` runs — no live LLM calls in Sections 1-5. Section 6 (optional) fires ONE live LangGraph node against OpenRouter to show the same audit trail on a different framework.

Everything runs from data shipped in this repo (`notebooks/data/`). Attendees can re-run every cell after the workshop.

## Colab setup (auto-skipped if running locally)

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q',
                    'agensflow-mcp', 'agensflow-langgraph',
                    'asgi-lifespan', 'langchain-openai', 'python-dotenv'], check=True)
    if not os.path.isdir('agensflow-langgraph'):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/agensflow-ai/agensflow-langgraph.git'], check=True)
    os.chdir('agensflow-langgraph/notebooks')
    print(f'  ✓ Colab ready — cwd = {os.getcwd()}')

## Setup

In [ ]:
import json, statistics
from pathlib import Path
from collections import defaultdict

# ────────────────────────────────────────────────────────────
# Change this to see the substrate settle on a different arm.
# 'thrifty' = cost-first, 'balanced' = default, 'premium' = quality-first
# ────────────────────────────────────────────────────────────
PROFILE = 'balanced'
MIN_N   = 5            # ignore arms with fewer than N observations

RUNS = Path('data/workshop_runs.jsonl')
rows = [json.loads(l) for l in RUNS.open()]
sigs = sorted({r['signature'] for r in rows})
print(f'  loaded {len(rows)} substrate runs')
print(f'  signatures       : {sigs}')
print(f'  active profile   : {PROFILE!r}')

## Section 1 — This is one substrate decision

Every routing decision is written to disk. Task → signature → per-stage tier choice → outcome (cost, latency, tokens, per-axis judge quality). Nothing hidden.

In [ ]:
# One representative accepted coord/opus row (top-tier at plan)
sample = next(r for r in rows if r['signature'] == 'coordination'
              and r['choice']['plan'] == 'opus/max'
              and r['final_acceptance'])

print(f'  task            : {sample["task"]}')
print(f'  signature       : {sample["signature"]}')
print(f'  choice:')
for stage, tier in sample['choice'].items():
    resolved = sample['choice_resolved'].get(stage)
    print(f'    {stage:<8}      → {tier:<15} ({resolved})')
print(f'  accepted        : {sample["final_acceptance"]}')
print(f'  cost            : ${sample["total_cost"]:.4f}')
print(f'  output tokens   : {sample["total_output_tokens"]:,}')
print(f'  latency         : {sample["total_latency"]:.1f}s')
print(f'  judge quality   : {sample["judge_quality"]:.3f}')
print(f'  reward ({PROFILE:<8}): {sample["reward_by_profile"][PROFILE]:+.3f}')
print(f'  per-axis (cand):')
for axis, val in sample['judge_detail']['axis_means_candidate'].items():
    print(f'    {axis:<14}  {val:.3f}')

> Say: "one substrate memory row. Every field queryable. Ask the substrate 'why did you pick this' and point at the reward table."

## Section 2 — Two patterns from three signatures

For each signature, top-3 action strings by mean reward across the sweep. The substrate found these from reward signal alone — no rules encoded the answer.

In [ ]:
def _reward(r):
    return (r.get('reward_by_profile') or {}).get(PROFILE)

by_sig_action = defaultdict(list)
for r in rows:
    rw = _reward(r)
    if rw is not None:
        by_sig_action[(r['signature'], r['action'])].append(rw)

for sig in ('coordination', 'simple', 'refactor'):
    print(f'\n{sig}:')
    entries = [(a, statistics.mean(rs), len(rs))
               for (s, a), rs in by_sig_action.items()
               if s == sig and len(rs) >= MIN_N]
    for a, mean_r, n in sorted(entries, key=lambda x: -x[1])[:3]:
        print(f'  {a:<52} μ={mean_r:+.3f}  n={n}')

> Say: "coord's top-3 all have opus at plan — substrate learned that class needs capability at planning. Simple and refactor top-3 are haiku everywhere. Two patterns emerged from three designed-as-different signatures, from reward signal alone."

## Section 3 — Token budgeting vs token maxing

For each signature: the substrate's learned argmax arm vs the "throw money at it" baseline (opus/max at plan + opus/medium at build+check — the top-spend arm actually available in the sweep). Same 780-run measurement conditions on both sides.

In [ ]:
# "Token maxing" baseline: the highest-tier config actually run in the sweep
# (plan:opus/max at every planning stage, opus/medium at build+check).
MAXING_ARM = 'plan:opus/max|build:opus/medium|check:opus/medium'

def _stats(rs):
    return {
        'n':    len(rs),
        'cost': statistics.mean(r['total_cost'] for r in rs),
        'q':    statistics.mean(r['judge_quality'] for r in rs),
    }

print(f'  {"signature":<14} {"maxing $":>10} {"learned $":>10} {"Δ cost":>8} '
      f'{"maxing q":>10} {"learned q":>10}   learned arm')
for sig in ('simple', 'refactor', 'coordination'):
    # substrate's argmax under active profile
    entries = [(a, statistics.mean(rs), len(rs))
               for (s, a), rs in by_sig_action.items()
               if s == sig and len(rs) >= MIN_N]
    if not entries: continue
    top_action = max(entries, key=lambda x: x[1])[0]
    learned = [r for r in rows if r['signature'] == sig
               and r['action'] == top_action and r['final_acceptance']]
    maxing  = [r for r in rows if r['signature'] == sig
               and r['action'] == MAXING_ARM and r['final_acceptance']]
    if not (learned and maxing): continue
    L, M = _stats(learned), _stats(maxing)
    delta = (L['cost'] - M['cost']) / M['cost'] * 100
    print(f'  {sig:<14} ${M["cost"]:>9.2f} ${L["cost"]:>9.2f} {delta:>+7.0f}% '
          f'{M["q"]:>10.3f} {L["q"]:>10.3f}   {top_action}')

> Say: "simple + refactor: substrate spends a fraction of the token-maxing baseline at similar-or-better judge quality. Coordination: substrate stays close on cost but with the same quality — because reward signal said the cheap arm doesn't hold there. This is workload-appropriate Pareto, not a cost heuristic."

## Section 4 — Same substrate, three profiles, three policies

This is what "budgeting" actually means: the SAME 780 runs, rescored by three reward profiles → three different argmax choices per signature. No retraining, no code change. The substrate learns one library of arm performance and lets each user dial the tradeoff.

In [ ]:
def _argmax_for(sig, prof, min_n=MIN_N):
    by_a = defaultdict(list)
    for r in rows:
        if r['signature'] != sig: continue
        rw = (r.get('reward_by_profile') or {}).get(prof)
        if rw is None: continue
        by_a[r['action']].append(rw)
    filtered = [(a, statistics.mean(rs), len(rs))
                for a, rs in by_a.items() if len(rs) >= min_n]
    return max(filtered, key=lambda x: x[1]) if filtered else None

for sig in ('coordination', 'simple', 'refactor'):
    print(f'\n{sig}:')
    picked = {}
    for prof in ('thrifty', 'balanced', 'premium'):
        result = _argmax_for(sig, prof)
        if not result: continue
        a, mu, n = result
        picked[prof] = a
        print(f'  {prof:<10} → {a:<52} μ={mu:+.3f} n={n}')
    distinct = len(set(picked.values()))
    print(f'  → {distinct} distinct arm(s) across profiles')

> Say: "same 780 runs, same substrate. Change the reward profile at import time and the argmax flips. Coord thrifty picks opus/medium at plan; premium picks opus/max. Simple thrifty picks haiku everywhere; premium adds an opus check pass. This isn't a per-profile trained model — it's per-profile *scoring* of one shared arm library. That's why it costs nothing to expose to end users."

## Section 5 — Warm-policy adaptation (new tier released mid-training)

Early chunks ran on an 18-arm action space. Later chunks expanded to 64 arms — opus at build+check, opus/max at plan. What did the substrate's argmax do?

In [ ]:
# Pre-expansion: rows where opus/max never appears in the chosen action
# Post-expansion: all rows (the opus/max arm is in the pool)
def _argmax_pre_post(sig):
    def _score(rs):
        by_a = defaultdict(list)
        for r in rs:
            rw = _reward(r)
            if rw is not None:
                by_a[r['action']].append(rw)
        filtered = [(a, statistics.mean(v), len(v)) for a, v in by_a.items() if len(v) >= MIN_N]
        return max(filtered, key=lambda x: x[1]) if filtered else ('—', 0.0, 0)

    pre  = _score([r for r in rows if r['signature'] == sig and 'opus/max' not in r['action']])
    post = _score([r for r in rows if r['signature'] == sig])
    return pre, post

for sig in ('coordination', 'simple', 'refactor'):
    (pre_a, pre_r, pre_n), (post_a, post_r, post_n) = _argmax_pre_post(sig)
    flipped = 'FLIPPED' if pre_a != post_a else 'HELD'
    print(f'\n  {sig} — {flipped}')
    print(f'    pre-expansion  → {pre_a:<52}  (n={pre_n})')
    print(f'    post-expansion → {post_a:<52}  (n={post_n})')

> Say: "Same substrate, no code change. When the action space expanded, coord flipped to opus/max because reward earned it. Simple TESTED opus/max and rejected — reward didn't back it. That's the enterprise-ask: 'when a new model releases, can I plug it in and let the system decide?' Yes, empirically."

## Section 6 — Same audit trail on a LangGraph node (live, OSS)

The 780-run sweep ran on a Claude Code coordination layer. Same audit trail works on ANY framework — here's one @agensflow-decorated LangGraph node routed through the exact same substrate, in this notebook.

**Requires**: `OPENROUTER_API_KEY` in env or a `.env` file. Cost: ~$0.02, ~10-15s. Skip if network/key unavailable — Sections 1-5 already tell the full story.

In [ ]:
import asyncio

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if not os.environ.get('OPENROUTER_API_KEY'):
    print('  skipped — set OPENROUTER_API_KEY to run this section.')
else:
    os.environ.setdefault('AGF_DATABASE_URL', 'sqlite+aiosqlite:///:memory:')
    os.environ.setdefault('AGF_ENV', 'test')
    os.environ.setdefault('AGF_JWT_SECRET', 'talk-demo')

    from httpx import ASGITransport, AsyncClient
    from asgi_lifespan import LifespanManager
    from agensflow_mcp.app import create_app
    from agensflow_mcp.db.session import init_db, get_engine
    from agensflow_mcp.db.models import Base

    app = create_app()
    await init_db()
    engine = get_engine()
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

    lifespan_mgr = LifespanManager(app)
    await lifespan_mgr.__aenter__()
    server_client = AsyncClient(transport=ASGITransport(app=app), base_url='http://test')

    resp = await server_client.post('/auth/anonymous')
    api_key = resp.json()['api_key']

    from agensflow_langgraph import client as agf_client
    class _NC(agf_client.AgensFlowClient):
        async def _a_post_model(self, path, payload, model_cls):
            r = await server_client.post(path, json=payload, headers=self._headers)
            self._raise_for_status(r); return model_cls.model_validate(r.json())
        async def _a_get_model(self, path, model_cls, params=None):
            r = await server_client.get(path, params=params, headers=self._headers)
            self._raise_for_status(r); return model_cls.model_validate(r.json())
    agf_client._CACHE.clear()
    agf_client.AgensFlowClient = _NC
    os.environ['AGENSFLOW_SERVER_URL'] = 'http://test'
    os.environ['AGENSFLOW_API_KEY'] = api_key

    from langchain_openai import ChatOpenAI
    from agensflow_langgraph import agensflow, arecord_reward

    def _or(mid):
        return ChatOpenAI(
            base_url='https://openrouter.ai/api/v1',
            api_key=os.environ['OPENROUTER_API_KEY'],
            model=mid, temperature=0.0, max_retries=2,
            default_headers={'HTTP-Referer': 'https://agensflow.ai',
                             'X-Title': 'AgensFlow workshop demo'},
        )

    @agensflow(pool={'cheap': _or('thinkingmachines/inkling'),
                     'deep':  _or('anthropic/claude-sonnet-5')})
    async def answer(state, model, config=None):
        msg = await model.ainvoke([('human', state['question'])])
        return {'answer': msg.content}

    r = await answer({'question': 'What does UCB1 optimize?'},
                     config={'configurable': {'thread_id': 'workshop_live_1'}})
    await arecord_reward(quality=0.9, thread_id='workshop_live_1')

    resp = await server_client.get('/langgraph/decisions?limit=1',
                                   headers={'Authorization': f'Bearer {api_key}'})
    print('  --- fresh decision, seconds old, same audit shape ---')
    print(json.dumps(resp.json()['decisions'][0], indent=2, default=str))

    await server_client.aclose()
    await lifespan_mgr.__aexit__(None, None, None)

> Say: "Same audit fields — arm chosen, cost, tokens, latency, quality. Not the 780-run Claude Code sweep — a LangGraph node in this notebook, seconds ago. Substrate treats them the same. Whatever coordination layer you build sits on top of the same auditability."

---

## Landing

> **Loop engineering** asks: how should the agent iterate?  
> **Harness engineering** asks: who decides what gets looped in, under which conditions, and with what budget?

**AgensFlow**: the harness that learns those decisions from outcome — and shows you exactly why each one was made.

- `pip install agensflow-mcp agensflow-langgraph`
- github.com/agensflow-ai/agensflow-langgraph